# 02 - Exploratory Data Analysis (EDA): MRL Eye Dataset

**Phase 1, step 2.** We explore the *cleaned* dataset from notebook 01 to understand class balance, subject/lighting/glasses effects, image properties and relationships between variables. Every figure is saved to `images/`, and the last section writes down the key findings automatically from the data.

Target variable: `closed` (1 = closed eye, 0 = open eye).

In [ ]:
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
from scipy.stats import chi2_contingency

sns.set_theme(style='whitegrid')
DATA_DIR = os.environ.get('MRL_DIR', '../data/raw/mrl')
PROC_DIR, IMG_DIR = '../data/processed', '../images'
os.makedirs(IMG_DIR, exist_ok=True)
def savefig(name): plt.savefig(os.path.join(IMG_DIR, name), dpi=150, bbox_inches='tight')

df = pd.read_csv(f'{PROC_DIR}/metadata_clean.csv')
df['label'] = df.closed.map({0: 'open', 1: 'closed'})
df['gender_name'] = df.gender.map({0: 'male', 1: 'female'})
df['glasses_name'] = df.glasses.map({0: 'no glasses', 1: 'glasses'})
df['lighting_name'] = df.lighting.map({0: 'bad light', 1: 'good light'})
df['reflection_name'] = df.reflections.map({0: 'none', 1: 'small', 2: 'big'})
df['sensor_name'] = 'sensor ' + df.sensor.astype(str)
print(df.shape); df.head()

## 1. Dataset overview

In [ ]:
print('Rows:', len(df), '| Subjects:', df.subject.nunique(), '| Features:', df.shape[1])
print('\nData types:'); print(df.dtypes.to_string())
print('\nNumeric summary:'); df[['width', 'height', 'file_kb', 'mean_intensity', 'std_intensity']].describe().round(2)

## 2. Class balance (target variable)
A heavily imbalanced target would make accuracy misleading and would call for class weights or resampling.

In [ ]:
counts = df.label.value_counts()
ratio = counts.max() / counts.min()
print(counts.to_string()); print(f'\nImbalance ratio (majority/minority): {ratio:.2f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=df, x='label', order=['open', 'closed'], palette=['#4c9f70', '#d9534f'], ax=ax[0])
for c in ax[0].containers: ax[0].bar_label(c)
ax[0].set_title('Class balance (count)')
ax[1].pie(counts[['open', 'closed']], labels=['open', 'closed'], autopct='%1.1f%%', colors=['#4c9f70', '#d9534f'])
ax[1].set_title('Class balance (share)')
plt.tight_layout(); savefig('eda_01_class_balance.png'); plt.show()

**What it tells us:** the bars/pie show how many open vs. closed eyes there are. If the ratio is close to 1 the dataset is balanced; otherwise use class weights and judge models with recall / F1 / AUC, not accuracy alone.

## 3. Metadata distributions
How are gender, glasses, reflections, lighting and sensor distributed? Skewed groups mean some conditions are under-represented.

In [ ]:
cats = ['gender_name', 'glasses_name', 'reflection_name', 'lighting_name', 'sensor_name']
fig, axes = plt.subplots(2, 3, figsize=(16, 8)); axes = axes.ravel()
for a, c in zip(axes, cats):
    sns.countplot(data=df, x=c, ax=a, color='steelblue', order=sorted(df[c].unique()))
    for cont in a.containers: a.bar_label(cont)
    a.set_title(c.replace('_name', '').title()); a.set_xlabel('')
axes[-1].axis('off')
plt.tight_layout(); savefig('eda_02_metadata_distributions.png'); plt.show()

**What it tells us:** which recording conditions dominate the data. A model trained on this will be best in the majority conditions, so under-represented ones (for example bad light or big reflections) deserve special attention during evaluation.

## 4. Class balance inside each condition
Is the open/closed ratio the same for every gender, glasses, reflection, lighting and sensor group? If not, those variables could leak label information or bias the model.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8)); axes = axes.ravel()
for a, c in zip(axes, cats):
    ct = pd.crosstab(df[c], df.label, normalize='index')[['open', 'closed']] * 100
    ct.plot.bar(stacked=True, ax=a, color=['#4c9f70', '#d9534f'], rot=0)
    a.axhline(50, color='k', ls='--', lw=.8); a.set_ylabel('% of images'); a.set_xlabel('')
    a.set_title(f"Open vs closed by {c.replace('_name', '')}")
axes[-1].axis('off')
plt.tight_layout(); savefig('eda_03_class_by_condition.png'); plt.show()

**What it tells us:** bars that stay near the dashed 50 % line mean the condition does not change the class ratio. Large departures would mean that variable is correlated with the label and could cause shortcut learning.

## 5. Subjects
MRL contains many images of each person. The images of one person look alike, so train/validation/test splits **must be done by subject**, otherwise the test score will be too optimistic.

In [ ]:
per_subj = df.groupby('subject').agg(n=('closed', 'size'), closed_ratio=('closed', 'mean'))
print(per_subj.describe().round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(per_subj.n, bins=25, ax=ax[0], color='slateblue'); ax[0].set_title('Images per subject'); ax[0].set_xlabel('images')
sns.histplot(per_subj.closed_ratio, bins=25, ax=ax[1], color='indianred'); ax[1].set_title('Share of closed eyes per subject')
ax[1].set_xlabel('closed ratio')
plt.tight_layout(); savefig('eda_04_subjects.png'); plt.show()

**What it tells us:** the left plot shows how unevenly images are spread over people; the right plot shows whether every subject has a similar open/closed mix. This justifies a subject-wise split in notebook 03.

## 6. Image properties: size, brightness, contrast

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(17, 8))
sns.histplot(data=df, x='mean_intensity', hue='label', bins=40, kde=True, palette=['#d9534f', '#4c9f70'], ax=ax[0, 0]); ax[0, 0].set_title('Brightness by class')
sns.histplot(data=df, x='std_intensity', hue='label', bins=40, kde=True, palette=['#d9534f', '#4c9f70'], ax=ax[0, 1]); ax[0, 1].set_title('Contrast by class')
sns.histplot(data=df, x='file_kb', bins=40, ax=ax[0, 2], color='gray'); ax[0, 2].set_title('File size (KB)')
sns.boxplot(data=df, x='label', y='mean_intensity', order=['open', 'closed'], palette=['#4c9f70', '#d9534f'], ax=ax[1, 0]); ax[1, 0].set_title('Brightness by class')
sns.boxplot(data=df, x='lighting_name', y='mean_intensity', hue='label', hue_order=['open', 'closed'], palette=['#4c9f70', '#d9534f'], ax=ax[1, 1]); ax[1, 1].set_title('Brightness by lighting and class')
sns.boxplot(data=df, x='sensor_name', y='mean_intensity', ax=ax[1, 2], color='lightsteelblue'); ax[1, 2].set_title('Brightness by sensor')
plt.tight_layout(); savefig('eda_05_brightness_contrast.png'); plt.show()

**What it tells us:** if the open and closed histograms overlap heavily, brightness alone cannot separate the classes (good, the model must look at the eye itself). Differences between lighting groups or sensors show how much the image appearance depends on the camera and light, which supports scaling the pixels and possibly augmenting brightness.

## 7. Correlation heatmap
All numeric variables, including the target `closed`.

In [ ]:
num_cols = ['closed', 'gender', 'glasses', 'reflections', 'lighting', 'sensor',
            'mean_intensity', 'std_intensity', 'width', 'height', 'file_kb']
corr = df[num_cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=.5)
plt.title('Correlation heatmap (Pearson)')
plt.tight_layout(); savefig('eda_06_correlation_heatmap.png'); plt.show()

print('Correlation of every variable with the target (closed):')
print(corr['closed'].drop('closed').sort_values(key=abs, ascending=False).round(3).to_string())

**What it tells us:** values near 0 in the `closed` row/column mean the metadata and simple image statistics carry almost no information about the label. Real information has to come from the eye pixels, so a CNN is appropriate. Strong correlations between two *features* (for example `width` and `height`) mean they are redundant.

## 8. Association of each categorical variable with the label (chi-square, Cramer's V)
Pearson correlation is not ideal for categories, so we also test independence and measure effect size.

In [ ]:
def cramers_v(a, b):
    ct = pd.crosstab(a, b)
    chi2, p, _, _ = chi2_contingency(ct)
    n = ct.values.sum()
    v = np.sqrt((chi2 / n) / max(min(ct.shape) - 1, 1))
    return v, p

rows = [(c.replace('_name', ''), *cramers_v(df[c], df.label)) for c in cats]
assoc = pd.DataFrame(rows, columns=['variable', 'cramers_v', 'p_value']).sort_values('cramers_v', ascending=False)
print(assoc.round(4).to_string(index=False))

plt.figure(figsize=(7, 3.5))
sns.barplot(data=assoc, y='variable', x='cramers_v', color='teal')
plt.title("Association with eye state (Cramer's V)"); plt.xlabel("Cramer's V (0 = none, 1 = perfect)")
plt.tight_layout(); savefig('eda_07_cramers_v.png'); plt.show()

**What it tells us:** a small Cramer's V (below about 0.1) means that variable barely relates to the label, even when the p-value is tiny (with so many images, almost everything is "significant").

## 9. Looking at the images
Sample grids per class and the **mean image** of each class (average of up to 1500 images per class, resized to 64x64).

In [ ]:
def load_gray(rel, size=64):
    return np.asarray(Image.open(os.path.join(DATA_DIR, rel)).convert('L').resize((size, size)), dtype=np.float32)

fig, ax = plt.subplots(2, 8, figsize=(16, 4.5))
for r, lab in enumerate(['open', 'closed']):
    sub = df[df.label == lab].sample(8, random_state=1)
    for c, rel in enumerate(sub.rel_path):
        ax[r, c].imshow(load_gray(rel), cmap='gray'); ax[r, c].axis('off')
    ax[r, 0].set_title(lab, loc='left', fontsize=12)
plt.suptitle('Random samples per class'); plt.tight_layout(); savefig('eda_08_sample_grid.png'); plt.show()

In [ ]:
means, pix = {}, {}
for lab in ['open', 'closed']:
    sub = df[df.label == lab].sample(min(1500, (df.label == lab).sum()), random_state=0)
    stack = np.stack([load_gray(r) for r in sub.rel_path])
    means[lab], pix[lab] = stack.mean(0), stack.ravel()

fig, ax = plt.subplots(1, 4, figsize=(17, 4))
ax[0].imshow(means['open'], cmap='gray'); ax[0].set_title('Mean image: open')
ax[1].imshow(means['closed'], cmap='gray'); ax[1].set_title('Mean image: closed')
im = ax[2].imshow(means['closed'] - means['open'], cmap='coolwarm'); ax[2].set_title('Difference (closed - open)')
plt.colorbar(im, ax=ax[2], fraction=.046)
for lab, col in [('open', '#4c9f70'), ('closed', '#d9534f')]:
    ax[3].hist(pix[lab], bins=50, alpha=.55, density=True, color=col, label=lab)
ax[3].set_title('Pixel-intensity distribution'); ax[3].legend()
for a in ax[:3]: a.axis('off')
plt.tight_layout(); savefig('eda_09_mean_images.png'); plt.show()

**What it tells us:** the mean images show the typical eye shape per class; the difference map highlights *where* open and closed eyes differ (the eyelid / pupil region). The intensity histograms show whether the pixel ranges of the two classes are similar and how wide the pixel range is, which informs scaling.

## 10. Key findings (generated from the data)
This cell computes the main findings from the numbers above and saves them to `images/key_findings.md`, so the text always matches the data. Copy the final text into the README ("Key findings").

In [ ]:
top = corr['closed'].drop('closed').abs().idxmax()
spread = per_subj.n.max() / max(per_subj.n.min(), 1)
b_open = df[df.closed == 0].mean_intensity.mean(); b_closed = df[df.closed == 1].mean_intensity.mean()
maj = counts.idxmax()

lines = [
 f'- The cleaned dataset has {len(df):,} images from {df.subject.nunique()} subjects.',
 f"- Class balance: {counts.get('open', 0):,} open vs {counts.get('closed', 0):,} closed "
 f"(majority = {maj}, imbalance ratio {ratio:.2f}). " +
 ('The data is close to balanced; plain accuracy is acceptable but still report recall and F1 for closed eyes.'
  if ratio < 1.2 else 'The data is imbalanced: use class weights and report recall/F1/AUC.'),
 f'- Images per subject range from {per_subj.n.min()} to {per_subj.n.max()} (max/min = {spread:.1f}x), '
 'so the split must be done by subject to avoid leakage.',
 f'- Gender / glasses / reflection / lighting / sensor are all included; the variable most associated with the label is '
 f"'{assoc.iloc[0].variable}' (Cramer's V = {assoc.iloc[0].cramers_v:.3f}).",
 f"- Strongest numeric correlation with the target: '{top}' (r = {corr.loc[top, 'closed']:.3f}). "
 'Metadata and global image statistics are weak predictors, so the model must learn from the eye pixels.',
 f'- Mean brightness: open {b_open:.1f} vs closed {b_closed:.1f} (difference {abs(b_open - b_closed):.1f} grey levels on a 0-255 scale).',
 f'- Image sizes vary (width {df.width.min()}-{df.width.max()} px, height {df.height.min()}-{df.height.max()} px) '
 f'and {df.brightness_outlier.sum():,} images are brightness outliers (kept, real lighting variation).',
 '- Pixel values span 0-255, so they should be scaled to [0, 1]; all images should be converted to grayscale and resized to one fixed size.',
 '- Modeling recommendations: subject-wise train/val/test split, class weights if imbalanced, brightness augmentation, '
 'evaluate per lighting / glasses group.',
]
text = '# Key findings from EDA\n\n' + '\n'.join(lines) + '\n'
open(f'{IMG_DIR}/key_findings.md', 'w', encoding='utf-8').write(text)
print(text)

## Figure guide (what each saved plot shows)

| File | Shows |
|---|---|
| `eda_01_class_balance.png` | Open vs closed counts and shares (class imbalance) |
| `eda_02_metadata_distributions.png` | How many images per gender, glasses, reflection, lighting, sensor |
| `eda_03_class_by_condition.png` | Whether the open/closed ratio changes between conditions |
| `eda_04_subjects.png` | Images per subject and closed share per subject (need for subject-wise split) |
| `eda_05_brightness_contrast.png` | Brightness and contrast distributions, boxplots by class / lighting / sensor |
| `eda_06_correlation_heatmap.png` | Linear correlations between all numeric variables and the target |
| `eda_07_cramers_v.png` | Strength of association between each category and the label |
| `eda_08_sample_grid.png` | Example images of both classes |
| `eda_09_mean_images.png` | Mean image per class, their difference and pixel histograms |

**Next:** `03_preprocessing.ipynb`